# CareerQuest (stretch): LoRA skill extractor

**Goal.** Teach a small instruction model to list the software and tools a job posting asks for, as a JSON array of O*NET names. This could later replace dictionary matching for skills that are phrased in unusual ways.

**Labels are weak.** `skills.jsonl` was labelled by CareerQuest's O*NET dictionary matcher, so the model first learns to imitate it. Report precision/recall against those labels, and hand-check 20 outputs for tools the dictionary missed. Treat this as an experiment, not a production component.

**Run on a GPU** (Colab T4 or Kaggle). Upload `skills.jsonl` from `data/processed/finetune/`. Licenses as in the embedder notebook: CC BY-SA 4.0 for anything derived from the postings.

In [ ]:
!pip install -q "transformers>=4.46" "peft>=0.13" "trl>=0.12" "datasets>=2.19" accelerate

In [ ]:
import json, os, pathlib, random, torch
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# Kaggle sets KAGGLE_KERNEL_RUN_TYPE; Colab also has a read-only /kaggle/input, so don't test for that.
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
DATA = pathlib.Path("/kaggle/input/careerquest-finetune") if ON_KAGGLE else pathlib.Path("finetune")
if not ON_KAGGLE:
    DATA.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import files
    for name, content in files.upload().items():
        (DATA / name).write_bytes(content)
except ImportError:
    pass
rows = [json.loads(line) for line in open(DATA / "skills.jsonl", encoding="utf-8")]
random.Random(13).shuffle(rows)
test, train = rows[:300], rows[300:]
print(len(train), "train,", len(test), "test")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
PROMPT = "List the software and tools this job posting asks for. Answer with a JSON array of names only.\n\nPosting:\n{text}"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto")

def messages(row, with_answer=True):
    chat = [{"role": "user", "content": PROMPT.format(text=row["text"])}]
    if with_answer:
        chat.append({"role": "assistant", "content": json.dumps(row["skills"])})
    return chat

def extract(rows, max_new_tokens=96):
    model.eval(); outputs = []
    for row in rows:
        ids = tokenizer.apply_chat_template(messages(row, False), add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        try:
            start, end = text.index("["), text.rindex("]") + 1
            outputs.append({s.strip() for s in json.loads(text[start:end]) if isinstance(s, str)})
        except ValueError:
            outputs.append(set())
    return outputs

def scores(pred, rows):
    tp = sum(len(p & set(r["skills"])) for p, r in zip(pred, rows))
    fp = sum(len(p - set(r["skills"])) for p, r in zip(pred, rows))
    fn = sum(len(set(r["skills"]) - p) for p, r in zip(pred, rows))
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(2 * precision * recall / max(precision + recall, 1e-9), 3)}

before = scores(extract(test[:100]), test[:100]); before

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

dataset = Dataset.from_list([{"messages": messages(r)} for r in train])
trainer = SFTTrainer(
    model=model, train_dataset=dataset, processing_class=tokenizer,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"),
    args=SFTConfig(output_dir="skill-lora-checkpoints", num_train_epochs=1, per_device_train_batch_size=8,
                   gradient_accumulation_steps=2, learning_rate=2e-4, logging_steps=25, save_strategy="no",
                   max_seq_length=768, bf16=torch.cuda.is_available(), report_to="none", seed=13),
)
trainer.train()
model = trainer.model

In [ ]:
predictions = extract(test[:100])
after = scores(predictions, test[:100])
print({"base": before, "lora": after})
for row, pred in list(zip(test, predictions))[:20]:   # hand-check: are the extras real tools?
    print(sorted(pred - set(row["skills"])), "| missed:", sorted(set(row["skills"]) - pred))

In [ ]:
trainer.model.save_pretrained("careerquest-skill-lora")
json.dump({"base_model": BASE_MODEL, "before": before, "after": after}, open("careerquest-skill-lora/metrics.json", "w"), indent=2)
!zip -qr careerquest-skill-lora.zip careerquest-skill-lora